## Scientific Validation

This notebook acts as a **scientific validation and documentation layer** on top of the already existing Machine Learning pipeline of the **Math-Music-Lab** project.

It does not replace the main Python files. Instead, it complements them:

- `math_music_data_cleaning.py` — data cleaning
- `math_music_eda_feature_engineering.py` — exploratory data analysis and feature engineering
- `math_music_modeling_and_mlflow.py` — modeling and MLflow tracking

The notebook focuses on aspects such as:

- whether the results are reproducible;
- where the data comes from and why it is suitable for this project;
- whether the target variable is properly defined;
- whether there is any risk of data leakage;
- comparison against a baseline model;
- model interpretation;
- error analysis;
- limitations of the project and possible improvements.

### Important Reproducibility Note

This notebook should be executed **at least two times**.

On the first run, it creates the following file:

`data/processed/data_version_manifest.json`

This file serves as the initial version record of the processed datasets.

On the second and later runs, the notebook compares the current processed datasets against the previously saved manifest. This makes it possible to check whether the data has remained unchanged or has been modified.

However, for convenience, the project will also include the necessary data required specifically to run this notebook. These files will be placed in:

`Math-Music-Lab/data/processed/` and `Math-Music-Lab/data/master_dataset_final.csv`

This allows the notebook to be executed directly, without requiring the user to manually prepare or generate all input files beforehand.

## Project Role and Scientific Framing

Main research question:

**Can the mathematical fingerprint of a song, represented through its audio features, help predict whether the song becomes a mainstream Billboard hit?**

In this project, each song is represented as a numerical vector of audio features:

$$
x = \text{audio features of a song}
$$

The target variable is a binary indicator:

$$
y =
\begin{cases}
1, & \text{if the song is a Billboard hit} \\
0, & \text{otherwise}
\end{cases}
$$

The goal of the model is to estimate whether the audio-feature vector contains useful information for predicting the probability that a song becomes a hit:

$$
f(x) \approx P(y = 1 \mid x)
$$

In simpler terms, the model tries to answer the question:

**Given only the measurable audio characteristics of a song, how likely is it that the song belongs to the Billboard-hit class?**

### Hypotheses

The **null hypothesis** is that the audio features do not contain enough useful predictive information.

If the null hypothesis is true, then we expect the model not to perform better than a very simple comparison model. For example, such a model could:

- choose the prediction randomly;
- always predict the most common result in the dataset, for example “not a hit”;
- be implemented with `DummyClassifier` from `scikit-learn`, which is commonly used as a simple baseline for comparison.

The **alternative hypothesis** is that the audio features do contain some useful information.

In other words, features such as tempo, energy, danceability, loudness, and others may help the model predict Billboard hits slightly better than such a simple baseline model.

### Scope and Limitations

This analysis is intentionally limited to audio-based information.

Billboard success depends on many factors that are not included in the dataset, such as:

- artist popularity;
- marketing and promotion;
- cultural timing;
- playlist placement;
- social media trends;
- genre popularity;
- radio exposure;
- collaborations and label support.

Because of this, the model should not be interpreted as a complete explanation of why a song becomes successful.

Instead, the goal is more modest:

**to test whether audio features alone contain a weak but measurable signal related to Billboard-hit prediction.**

In [ ]:
print("=" * 80)
print("Scientific Project Check")
print("=" * 80)
print("This file extends the existing pipeline with reproducibility, validation,")
print("interpretability, and error analysis.")
print("=" * 80)

## 1. Imports, Configuration, and Reproducibility Checks

This section verifies that the expected processed datasets exist and records key reproducibility information: paths, file checksums, dataset shapes, target distribution, package versions, and random seed.

It also creates or compares the data-version manifest file:

`data/processed/data_version_manifest.json`

On the first run, the manifest is created. On later runs, the current datasets are compared against the saved baseline.

In [ ]:
import sys
import platform
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

import sklearn

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path(".").resolve()

MASTER_DATASET_PROCESSED = PROJECT_ROOT / "data" / "processed" / "master_dataset.csv"
MASTER_DATASET_FINAL = PROJECT_ROOT / "data" / "master_dataset_final.csv"
CORGIS_HISTORICAL_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corgis_historical.csv"

REQUIRED_FILES = {
    "master_dataset_processed": MASTER_DATASET_PROCESSED,
    "master_dataset_final": MASTER_DATASET_FINAL,
    "corgis_historical_processed": CORGIS_HISTORICAL_PROCESSED,
}


def compute_file_checksum(path, algorithm="sha256"):
    """
    Compute a checksum for a file.

    The checksum is used as a simple data-versioning mechanism.
    If the source URLs change or the data is regenerated differently,
    the checksum will change as well.
    """
    hash_object = hashlib.new(algorithm)

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            hash_object.update(chunk)

    return hash_object.hexdigest()


print("\n" + "=" * 80)
print("1. Reproducibility checks")
print("=" * 80)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Random seed: {RANDOM_SEED}")

print(
    "\nEnvironment note: these versions are recorded for reproducibility. "
    "For a submitted repository, the same dependencies should also be listed "
    "in requirements.txt."
)

print("\nChecking required result files:")

missing_files = []

for name, path in REQUIRED_FILES.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        checksum = compute_file_checksum(path)
        print(f"  OK: {name}")
        print(f"      Path: {path}")
        print(f"      Size: {size_mb:.2f} MB")
        print(f"      SHA256: {checksum[:16]}...")
    else:
        print(f"  MISSING: {name}")
        print(f"      Expected path: {path}")
        missing_files.append(path)

if missing_files:
    raise FileNotFoundError(
        "Some required files are missing. "
        "Please run the original three scripts before running this validation file."
    )

print("\nAll required result files are available.")

df_master = pd.read_csv(MASTER_DATASET_PROCESSED)
df_final = pd.read_csv(MASTER_DATASET_FINAL)
df_corgis = pd.read_csv(CORGIS_HISTORICAL_PROCESSED)

print("\nLoaded datasets:")
print(f"  df_master: {df_master.shape}")
print(f"  df_final:  {df_final.shape}")
print(f"  df_corgis: {df_corgis.shape}")

if "is_hit" not in df_master.columns:
    raise ValueError("Column 'is_hit' is missing from df_master.")

if "is_hit" not in df_final.columns:
    raise ValueError("Column 'is_hit' is missing from df_final.")

print("\nTarget distribution in df_master:")
print(df_master["is_hit"].value_counts(dropna=False))
print(f"Hit rate: {df_master['is_hit'].mean() * 100:.2f}%")

print("\nTarget distribution in df_final:")
print(df_final["is_hit"].value_counts(dropna=False))
print(f"Hit rate: {df_final['is_hit'].mean() * 100:.2f}%")

BASE_AUDIO_FEATURES = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

missing_audio_features = [
    col for col in BASE_AUDIO_FEATURES if col not in df_master.columns
]

if missing_audio_features:
    raise ValueError(
        f"The following expected audio features are missing: {missing_audio_features}"
    )

print("\nBase audio features available:")
for col in BASE_AUDIO_FEATURES:
    print(f"  - {col}")

print("\nReproducibility check complete.")

In [ ]:
import json
from datetime import datetime, timezone

DATA_VERSION_MANIFEST = PROJECT_ROOT / "data" / "processed" / "data_version_manifest.json"


def summarize_dataset_for_manifest(name, path, dataframe):
    """
    Create a compact reproducibility summary for a dataset.

    This summary is stored in a JSON manifest and can be compared
    across different runs of the project.
    """
    summary = {
        "name": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "rows": int(dataframe.shape[0]),
        "columns_count": int(dataframe.shape[1]),
        "columns": list(dataframe.columns),
        "sha256": compute_file_checksum(path),
        "file_size_bytes": int(path.stat().st_size),
    }

    if "is_hit" in dataframe.columns:
        value_counts = dataframe["is_hit"].value_counts(dropna=False).to_dict()

        summary["target_is_hit"] = {
            "value_counts": {str(k): int(v) for k, v in value_counts.items()},
            "hit_rate": float(dataframe["is_hit"].mean()),
        }

    return summary


def build_current_manifest():
    """
    Build the current data-version manifest.

    The manifest records:
    - dataset checksums
    - dataset shapes
    - column names
    - target distribution
    - package versions
    - random seed
    """
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "environment": {
            "python_version": sys.version.split()[0],
            "platform": platform.platform(),
            "numpy_version": np.__version__,
            "pandas_version": pd.__version__,
            "sklearn_version": sklearn.__version__,
            "random_seed": RANDOM_SEED,
        },
        "datasets": {
            "master_dataset_processed": summarize_dataset_for_manifest(
                "master_dataset_processed",
                MASTER_DATASET_PROCESSED,
                df_master,
            ),
            "master_dataset_final": summarize_dataset_for_manifest(
                "master_dataset_final",
                MASTER_DATASET_FINAL,
                df_final,
            ),
            "corgis_historical_processed": summarize_dataset_for_manifest(
                "corgis_historical_processed",
                CORGIS_HISTORICAL_PROCESSED,
                df_corgis,
            ),
        },
    }

    return manifest


def compare_dataset_summaries(previous_summary, current_summary):
    """
    Compare one dataset summary from the previous manifest
    with the current dataset summary.
    """
    differences = []

    fields_to_compare = [
        "sha256",
        "rows",
        "columns_count",
        "columns",
        "file_size_bytes",
    ]

    for field in fields_to_compare:
        if previous_summary.get(field) != current_summary.get(field):
            differences.append(field)

    previous_target = previous_summary.get("target_is_hit")
    current_target = current_summary.get("target_is_hit")

    if previous_target != current_target:
        differences.append("target_is_hit")

    return differences


def compare_manifests(previous_manifest, current_manifest):
    """
    Compare previous and current data manifests.

    Returns a dictionary describing which datasets changed.
    """
    comparison = {}

    previous_datasets = previous_manifest.get("datasets", {})
    current_datasets = current_manifest.get("datasets", {})

    for dataset_name, current_summary in current_datasets.items():
        if dataset_name not in previous_datasets:
            comparison[dataset_name] = {
                "status": "NEW",
                "differences": ["dataset_missing_in_previous_manifest"],
            }
            continue

        previous_summary = previous_datasets[dataset_name]
        differences = compare_dataset_summaries(previous_summary, current_summary)

        if differences:
            comparison[dataset_name] = {
                "status": "CHANGED",
                "differences": differences,
            }
        else:
            comparison[dataset_name] = {
                "status": "UNCHANGED",
                "differences": [],
            }

    return comparison


def print_manifest_comparison(comparison):
    """
    Print a readable comparison report.
    """
    print("\n" + "=" * 80)
    print("Data version manifest comparison")
    print("=" * 80)

    changed_anything = False

    for dataset_name, result in comparison.items():
        status = result["status"]
        differences = result["differences"]

        print(f"\n{dataset_name}: {status}")

        if differences:
            changed_anything = True
            print("  Differences:")
            for diff in differences:
                print(f"    - {diff}")

    print("\nConclusion:")

    if changed_anything:
        print("At least one dataset changed compared to the previous manifest.")
        print("Model results may differ from the previous run.")
    else:
        print("All datasets match the previous recorded manifest.")
        print(
            "Results should be reproducible or very close, "
            "provided that code, seed, and package versions are unchanged."
        )


current_manifest = build_current_manifest()

if DATA_VERSION_MANIFEST.exists():
    print("\nExisting data version manifest found.")
    print(f"Manifest path: {DATA_VERSION_MANIFEST}")

    with open(DATA_VERSION_MANIFEST, "r", encoding="utf-8") as file:
        previous_manifest = json.load(file)

    manifest_comparison = compare_manifests(previous_manifest, current_manifest)
    print_manifest_comparison(manifest_comparison)
    print(
        "\nManifest note: the existing manifest is treated as the baseline and is "
        "not overwritten automatically."
    )

else:
    print("\nNo previous data version manifest found.")
    print("Creating baseline data version manifest.")

    DATA_VERSION_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

    with open(DATA_VERSION_MANIFEST, "w", encoding="utf-8") as file:
        json.dump(current_manifest, file, indent=2, ensure_ascii=False)

    print(f"Created: {DATA_VERSION_MANIFEST}")
    print("This file will be used as the baseline for future reproducibility checks.")

## 2. Data Source Documentation and Data Dictionary

This section explains the datasets used in the project and how they fit into the overall Math-Music-Lab pipeline.

It describes the role of each dataset, the main audio features, and why Spotify-style audio variables can be linked to Billboard chart outcomes for a supervised learning task.

The goal is to make the data sources clear: what each file represents, how it is used, and why it is relevant for predicting whether a song becomes a Billboard hit.

In [ ]:
print("\n" + "=" * 80)
print("2. Data source documentation and data dictionary")
print("=" * 80)

DATASET_DOCUMENTATION = pd.DataFrame(
    [
        {
            "dataset": "master_dataset_processed",
            "source_role": "Spotify audio features + Billboard hit target",
            "main_use": "Supervised classification and regression",
            "target_available": "yes",
        },
        {
            "dataset": "master_dataset_final",
            "source_role": "Feature-engineered analytical dataset",
            "main_use": "EDA, visualization, diagnostics, cautious modeling",
            "target_available": "yes",
        },
        {
            "dataset": "corgis_historical_processed",
            "source_role": "Historical/contextual music data",
            "main_use": "Time-series and background analysis",
            "target_available": "no / not primary",
        },
    ]
)

print("\nDataset documentation:")
print(DATASET_DOCUMENTATION.to_string(index=False))

PIPELINE_DOCUMENTATION = pd.DataFrame(
    [
        {
            "step": "1_data_cleaning",
            "script": "math_music_data_cleaning.py",
            "main_inputs": "raw Spotify/CORGIS/Billboard data sources",
            "main_outputs": "master_dataset.csv, corgis_historical.csv, cleaned CSV files",
            "role": "downloads, cleans, standardizes, merges sources, creates is_hit",
        },
        {
            "step": "2_eda_feature_engineering",
            "script": "math_music_eda_feature_engineering.py",
            "main_inputs": "cleaned master and CORGIS datasets",
            "main_outputs": "master_dataset_final.csv",
            "role": "EDA, feature engineering, scaling, PCA, clustering, time-series analysis",
        },
        {
            "step": "3_modeling_mlflow",
            "script": "math_music_modeling_and_mlflow.py",
            "main_inputs": "master_dataset_final.csv",
            "main_outputs": "MLflow runs, metrics, model diagnostics",
            "role": "initial regression/classification modeling and experiment tracking",
        },
        {
            "step": "4_scientific_validation",
            "script": "math_music_scientific_validation.py",
            "main_inputs": "processed/final datasets produced by earlier scripts",
            "main_outputs": "reproducibility, leakage, CV, interpretation, and error-analysis reports",
            "role": "scientific validation layer that addresses evaluator feedback",
        },
    ]
)

print("\nPipeline documentation:")
print(PIPELINE_DOCUMENTATION.to_string(index=False))

FEATURE_DICTIONARY = pd.DataFrame(
    [
        {
            "feature": "danceability",
            "meaning": "How suitable a track is for dancing",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "energy",
            "meaning": "Perceptual intensity and activity",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "loudness",
            "meaning": "Overall loudness in decibels",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "speechiness",
            "meaning": "Presence of spoken words",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "acousticness",
            "meaning": "Confidence that the track is acoustic",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "instrumentalness",
            "meaning": "Likelihood that the track has no vocals",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "liveness",
            "meaning": "Presence of live performance characteristics",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "valence",
            "meaning": "Musical positiveness or mood",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "tempo",
            "meaning": "Estimated beats per minute",
            "ml_role": "safe audio feature",
        },
        {
            "feature": "is_hit",
            "meaning": "Binary Billboard hit indicator",
            "ml_role": "target variable",
        },
    ]
)

print("\nMain feature dictionary:")
print(FEATURE_DICTIONARY.to_string(index=False))

CAUTION_COLUMNS = [
    col
    for col in ["PC1", "PC2", "cluster", "peak_pos", "wks_on_chart"]
    if col in df_final.columns
]

print("\nColumns requiring caution:")
if CAUTION_COLUMNS:
    for col in CAUTION_COLUMNS:
        print(f"  - {col}")
else:
    print("  No predefined caution columns found in df_final.")

SOURCE_LINKAGE_DOCUMENTATION = pd.DataFrame(
    [
        {
            "source": "Spotify-style audio features",
            "represents": "Internal acoustic/mathematical song structure",
            "examples": "tempo, loudness, energy, valence, danceability",
            "ml_role": "input features X",
        },
        {
            "source": "Billboard chart data",
            "represents": "External market/chart outcome",
            "examples": "chart presence, hit indicator",
            "ml_role": "target variable y",
        },
        {
            "source": "CORGIS historical music data",
            "represents": "Historical/contextual music trends",
            "examples": "year-level or historical descriptors",
            "ml_role": "contextual analysis, not primary target",
        },
    ]
)

print("\nSource linkage documentation:")
print(SOURCE_LINKAGE_DOCUMENTATION.to_string(index=False))

print("\nSource linkage interpretation:")
print(
    "Spotify-style audio variables provide the numerical input representation, "
    "while Billboard data provides the external success label."
)
print(
    "The merge is scientifically useful because it allows the project to test "
    "whether audio structure has measurable predictive signal for chart success."
)
print(
    "The merge is also limited because many non-audio causes of success are "
    "not represented in the feature matrix."
)

print("\nData source documentation complete.")